# Xenium 3-Head Training

### 3-Head Architecture
- **Head A: Presence** — P(gene expressed) — Binary classification (BCE)
- **Head B: Expression** — E[expr | expressed] — Conditional regression (masked SmoothL1 + PCC)
- **Head C: Uncertainty** — σ(calibrated) — Predicts expected absolute error

### Loss Design (Decoupled — 모든 항 ≥ 0)
```
L_total = α × BCE(presence)
        + β × masked_SmoothL1(expression)
        + δ × (1 - PCC(expression))    ← per-gene correlation loss
        + γ × SmoothL1(σ, |error|)     ← uncertainty calibration
```

### Final Prediction
```
y_pred = sigmoid(presence_logits) × expression_mu
uncertainty = softplus(raw_sigma)  # per-gene calibrated σ ≈ expected |error|
```

In [ ]:
import os
import random
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ===== Config =====
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

data_dir = '../../data/marker_gene_spatial_expression'
model_path = '../../model/marker_gene_spatial_expression/Xenium/'
os.makedirs(model_path, exist_ok=True)

NUM_GENES = 19
BATCH_SIZE = 16
NUM_EPOCHS = 1000
LR = 2e-4
NUM_WORKERS = 4
ENCODER_NAME = 'tu-convnext_tiny'

MARKER_GENES = [
    # Epithelial
    "EPCAM", "EGFR",
    # Fibroblast / Stromal
    "ACTA2", "PDGFRA", "PDGFRB", "SFRP4",
    # Endothelial
    "PECAM1",
    # Macrophage / Myeloid
    "CD68", "AIF1", "FCGR3A", "MRC1",
    # T cell
    "CD3E", "CD4", "CD8A", "TRAC",
    # B cell
    "CD79A", "MS4A1", "BANK1", "TCL1A"
]

In [ ]:
# ===== Augmentation =====
def apply_color_augmentation(image):
    """H&E 병리 이미지용 color augmentation (numpy uint8 HWC)"""
    if random.random() < 0.5:
        f = random.uniform(0.8, 1.2)
        image = np.clip(image * f, 0, 255).astype(np.uint8)
    if random.random() < 0.5:
        f = random.uniform(0.8, 1.2)
        mean = image.mean()
        image = np.clip((image - mean) * f + mean, 0, 255).astype(np.uint8)
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = np.clip(hsv[:, :, 0] + random.uniform(-10, 10), 0, 179)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.7, 1.3), 0, 255)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    if random.random() < 0.3:
        inv_gamma = 1.0 / random.uniform(0.8, 1.2)
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype(np.uint8)
        image = cv2.LUT(image, table)
    if random.random() < 0.3:
        for c in range(3):
            image[:, :, c] = np.clip(
                image[:, :, c].astype(np.float32) + random.uniform(-10, 10), 0, 255
            ).astype(np.uint8)
    if random.random() < 0.3:
        noise = np.random.normal(0, random.uniform(0, 5), image.shape)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < 0.2:
        ks = random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ks, ks), 0)
    return image


# ===== Dataset =====
class ThreeHeadDataset(Dataset):
    def __init__(self, img_20x_list, img_5x_list, label_pres_list, label_expr_list, augment=False):
        self.img_20x_paths = img_20x_list
        self.img_5x_paths = img_5x_list
        self.label_pres_paths = label_pres_list
        self.label_expr_paths = label_expr_list
        self.augment = augment
        self.to_tensor = ToTensor()

    def __len__(self):
        return len(self.img_20x_paths)

    def __getitem__(self, idx):
        img_20x = np.array(Image.open(self.img_20x_paths[idx]).convert('RGB'))
        img_5x = np.array(Image.open(self.img_5x_paths[idx]).convert('RGB'))
        label_pres = np.load(self.label_pres_paths[idx]).astype(np.float32)
        label_expr = np.load(self.label_expr_paths[idx]).astype(np.float32)

        label_pres = np.nan_to_num(label_pres, nan=0.0)
        label_expr = np.nan_to_num(label_expr, nan=0.0)
        label_pres = np.clip(label_pres, 0.0, 1.0)
        label_expr = np.clip(label_expr, 0.0, 1.0)

        label_pres = torch.from_numpy(label_pres)
        label_expr = torch.from_numpy(label_expr)

        if self.augment:
            if random.random() < 0.5:
                img_20x = np.fliplr(img_20x).copy()
                img_5x = np.fliplr(img_5x).copy()
            if random.random() < 0.5:
                img_20x = np.flipud(img_20x).copy()
                img_5x = np.flipud(img_5x).copy()
            if random.random() < 0.3:
                k = random.randint(1, 3)
                img_20x = np.rot90(img_20x, k).copy()
                img_5x = np.rot90(img_5x, k).copy()
            img_20x = apply_color_augmentation(img_20x)
            img_5x = apply_color_augmentation(img_5x)

        img_20x = self.to_tensor(img_20x.copy())
        img_5x = self.to_tensor(img_5x.copy())

        return img_20x, img_5x, label_pres, label_expr


# ===== File list =====
img_20x_list = sorted(glob(f'{data_dir}/image/Xenium/**/*.tiff', recursive=True))
print(f'Total 20x patches: {len(img_20x_list)}')

img_5x_list = [p.replace('/image/', '/image_5x/') for p in img_20x_list]
label_pres_list = [p.replace('/image/', '/label_presence/').replace('.tiff', '.npy') for p in img_20x_list]
label_expr_list = [p.replace('/image/', '/label_expression/').replace('.tiff', '.npy') for p in img_20x_list]

# 존재하지 않는 파일 필터링
valid_idx = [i for i in range(len(img_20x_list))
             if os.path.exists(img_5x_list[i])
             and os.path.exists(label_pres_list[i])
             and os.path.exists(label_expr_list[i])]
print(f'Valid samples: {len(valid_idx)} / {len(img_20x_list)}')

img_20x_list = [img_20x_list[i] for i in valid_idx]
img_5x_list = [img_5x_list[i] for i in valid_idx]
label_pres_list = [label_pres_list[i] for i in valid_idx]
label_expr_list = [label_expr_list[i] for i in valid_idx]

# Train / Val split
indices = list(range(len(img_20x_list)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

train_dataset = ThreeHeadDataset(
    [img_20x_list[i] for i in train_idx], [img_5x_list[i] for i in train_idx],
    [label_pres_list[i] for i in train_idx], [label_expr_list[i] for i in train_idx],
    augment=True
)
val_dataset = ThreeHeadDataset(
    [img_20x_list[i] for i in val_idx], [img_5x_list[i] for i in val_idx],
    [label_pres_list[i] for i in val_idx], [label_expr_list[i] for i in val_idx],
    augment=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# ===== Model: 3-Head Architecture =====
class MultiScaleModel3Head(nn.Module):
    """
    3-Head Model:
      Head A: Presence  — P(gene expressed in patch) — logits for BCE
      Head B: Expression — E[expr | expressed] — conditional mean (sigmoid → [0,1])
      Head C: Uncertainty — log(variance) — aleatoric uncertainty
    """
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=19):
        super().__init__()
        self.encoder_20x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')
        self.encoder_5x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')

        enc_channels = self.encoder_20x.out_channels
        feat_dim = enc_channels[-1]  # 768 for ConvNeXt-Tiny

        # Shared projection after fusion
        self.shared_fc = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

        # Head A: Presence (logits → BCE)
        self.head_presence = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_genes),
        )

        # Head B: Expression mean (conditional, sigmoid → [0,1])
        self.head_expression = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_genes),
            nn.Sigmoid(),
        )

        # Head C: Uncertainty (log variance, unconstrained)
        self.head_log_var = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_genes),
        )

    def forward(self, x_20x, x_5x):
        feat_20x = self.encoder_20x(x_20x)
        feat_5x = self.encoder_5x(x_5x)

        # Fuse last stage features
        fused = feat_20x[-1] + feat_5x[-1]  # (B, 768, H, W)

        # Global Average Pooling → shared projection
        pooled = F.adaptive_avg_pool2d(fused, 1).flatten(1)  # (B, 768)
        shared = self.shared_fc(pooled)  # (B, 512)

        presence_logits = self.head_presence(shared)    # (B, G) raw logits
        expression_mu = self.head_expression(shared)    # (B, G) [0, 1]
        log_var = self.head_log_var(shared)             # (B, G) unconstrained

        return presence_logits, expression_mu, log_var


# ===== Loss: Decoupled 3-Head + PCC =====
class ThreeHeadLoss(nn.Module):
    """
    Decoupled 3-Head Loss + PCC (모든 항이 non-negative):
      Head A: BCEWithLogitsLoss — presence detection
      Head B: Masked SmoothL1 + PCC Loss — expression regression
      Head C: Calibration SmoothL1 — σ가 |error|를 예측하도록 학습

    total = α × BCE + β × SmoothL1(expr) + δ × (1 - PCC(expr)) + γ × SmoothL1(σ, |error|)
    """
    def __init__(self, presence_weight=1.0, expression_weight=1.0, pcc_weight=0.5, uncertainty_weight=0.5):
        super().__init__()
        self.pw = presence_weight
        self.ew = expression_weight
        self.pcw = pcc_weight
        self.uw = uncertainty_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, presence_logits, expr_mu, raw_sigma, gt_presence, gt_expression):
        gt_presence = gt_presence.clamp(0.0, 1.0)
        gt_expression = gt_expression.clamp(0.0, 1.0)

        # Head A: Presence loss (BCE — all genes, all samples)
        loss_pres = self.bce(presence_logits, gt_presence)

        # Mask: only where gene is actually present
        mask = gt_presence > 0.5  # (B, G)

        if mask.sum() > 0:
            # Head B-1: Expression regression (masked SmoothL1)
            loss_expr = F.smooth_l1_loss(expr_mu[mask], gt_expression[mask])

            # Head B-2: PCC Loss (per-gene correlation across batch)
            pred_c = expr_mu - expr_mu.mean(dim=0, keepdim=True)
            gt_c = gt_expression - gt_expression.mean(dim=0, keepdim=True)
            cov = (pred_c * gt_c).sum(dim=0)
            pred_std = pred_c.pow(2).sum(dim=0).sqrt()
            gt_std = gt_c.pow(2).sum(dim=0).sqrt()
            pcc = cov / (pred_std * gt_std + 1e-8)
            loss_pcc = 1.0 - pcc.mean()

            # Head C: Uncertainty calibration
            sigma = F.softplus(raw_sigma)
            abs_error = (expr_mu - gt_expression).abs().detach()
            loss_unc = F.smooth_l1_loss(sigma[mask], abs_error[mask])
        else:
            loss_expr = torch.tensor(0.0, device=presence_logits.device)
            loss_pcc = torch.tensor(0.0, device=presence_logits.device)
            loss_unc = torch.tensor(0.0, device=presence_logits.device)

        total = self.pw * loss_pres + self.ew * loss_expr + self.pcw * loss_pcc + self.uw * loss_unc
        return total, loss_pres, loss_expr, loss_pcc, loss_unc


# ===== PCC metric =====
def pearson_corr_vector(pred, target):
    """Per-gene Pearson correlation. (B, G) -> (G,)"""
    pred_c = pred - pred.mean(dim=0, keepdim=True)
    target_c = target - target.mean(dim=0, keepdim=True)
    cov = (pred_c * target_c).sum(dim=0)
    pred_std = pred_c.pow(2).sum(dim=0).sqrt()
    target_std = target_c.pow(2).sum(dim=0).sqrt()
    pcc = cov / (pred_std * target_std + 1e-8)
    return pcc


# ===== Init =====
model = MultiScaleModel3Head(ENCODER_NAME, NUM_GENES).to(device)
criterion = ThreeHeadLoss(presence_weight=1.0, expression_weight=1.0, pcc_weight=0.5, uncertainty_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')
print(f'Loss: BCE(presence) + SmoothL1(expr) + PCC(expr) + calibration(uncertainty)')

# Load checkpoint
ckpt_path = os.path.join(model_path, 'train_3head_best.pt')
if os.path.exists(ckpt_path):
    state_dict = torch.load(ckpt_path, map_location=device)
    has_nan = any(torch.isnan(v).any().item() for v in state_dict.values() if v.is_floating_point())
    has_inf = any(torch.isinf(v).any().item() for v in state_dict.values() if v.is_floating_point())
    if has_nan or has_inf:
        print(f'[WARNING] Checkpoint contains NaN/Inf! Training from scratch.')
    else:
        model.load_state_dict(state_dict, strict=False)
        print(f'Checkpoint loaded: {ckpt_path}')
else:
    print('No checkpoint found. Training from scratch.')

# Shape test
with torch.no_grad():
    dummy_20x = torch.randn(2, 3, 512, 512).to(device)
    dummy_5x = torch.randn(2, 3, 512, 512).to(device)
    p_logits, e_mu, lv = model(dummy_20x, dummy_5x)
    print(f'Output shapes: presence={p_logits.shape}, expression={e_mu.shape}, log_var={lv.shape}')
    print(f'presence range: [{p_logits.min():.3f}, {p_logits.max():.3f}]')
    print(f'expression range: [{e_mu.min():.3f}, {e_mu.max():.3f}]')
    print(f'log_var range: [{lv.min():.3f}, {lv.max():.3f}]')

In [ ]:
# ===== Training Loop =====
train_loss_list, val_loss_list = [], []
train_pres_loss_list, val_pres_loss_list = [], []
train_expr_loss_list, val_expr_loss_list = [], []
train_pcc_pres_list, val_pcc_pres_list = [], []
train_pcc_expr_list, val_pcc_expr_list = [], []
train_mean_unc_list, val_mean_unc_list = [], []
train_pcc_loss_list, val_pcc_loss_list = [], []
train_unc_loss_list, val_unc_loss_list = [], []
MIN_loss = float('inf')
MAX_GRAD_NORM = 1.0

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running_loss, running_pres, running_expr, running_pcc, running_unc = 0, 0, 0, 0, 0
    all_pred_pres, all_gt_pres = [], []
    all_pred_expr, all_gt_expr = [], []
    all_uncertainty = []
    n_valid_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:4d} [Train]', leave=False)
    for img_20x, img_5x, gt_pres, gt_expr in pbar:
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        gt_pres = gt_pres.to(device)
        gt_expr = gt_expr.to(device)

        optimizer.zero_grad()
        pres_logits, expr_mu, log_var = model(img_20x, img_5x)
        loss, lp, le, lpc, lu = criterion(pres_logits, expr_mu, log_var, gt_pres, gt_expr)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  [!] NaN/Inf loss detected, skipping batch')
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

        n_valid_batches += 1
        running_loss += loss.item()
        running_pres += lp.item()
        running_expr += le.item()
        running_pcc += lpc.item()
        running_unc += lu.item()

        all_pred_pres.append(torch.sigmoid(pres_logits).detach().cpu())
        all_gt_pres.append(gt_pres.cpu())
        all_pred_expr.append(expr_mu.detach().cpu())
        all_gt_expr.append(gt_expr.cpu())
        all_uncertainty.append(F.softplus(log_var).detach().cpu())

        pbar.set_postfix(loss=f'{running_loss / n_valid_batches:.4f}')

    n_train = max(n_valid_batches, 1)
    train_loss_list.append(running_loss / n_train)
    train_pres_loss_list.append(running_pres / n_train)
    train_expr_loss_list.append(running_expr / n_train)
    train_pcc_loss_list.append(running_pcc / n_train)
    train_unc_loss_list.append(running_unc / n_train)

    if all_pred_pres:
        all_pred_pres_t = torch.cat(all_pred_pres)
        all_gt_pres_t = torch.cat(all_gt_pres)
        all_pred_expr_t = torch.cat(all_pred_expr)
        all_gt_expr_t = torch.cat(all_gt_expr)
        all_unc_t = torch.cat(all_uncertainty)
        train_pcc_pres_list.append(pearson_corr_vector(all_pred_pres_t, all_gt_pres_t).mean().item())
        train_pcc_expr_list.append(pearson_corr_vector(all_pred_expr_t, all_gt_expr_t).mean().item())
        train_mean_unc_list.append(all_unc_t.mean().item())
    else:
        train_pcc_pres_list.append(0.0)
        train_pcc_expr_list.append(0.0)
        train_mean_unc_list.append(0.0)

    # --- Validation ---
    model.eval()
    running_loss, running_pres, running_expr, running_pcc, running_unc = 0, 0, 0, 0, 0
    all_pred_pres, all_gt_pres = [], []
    all_pred_expr, all_gt_expr = [], []
    all_uncertainty = []

    with torch.no_grad():
        for img_20x, img_5x, gt_pres, gt_expr in tqdm(val_loader, desc=f'Epoch {epoch:4d} [Val]', leave=False):
            img_20x = img_20x.to(device)
            img_5x = img_5x.to(device)
            gt_pres = gt_pres.to(device)
            gt_expr = gt_expr.to(device)

            pres_logits, expr_mu, log_var = model(img_20x, img_5x)
            loss, lp, le, lpc, lu = criterion(pres_logits, expr_mu, log_var, gt_pres, gt_expr)

            running_loss += loss.item()
            running_pres += lp.item()
            running_expr += le.item()
            running_pcc += lpc.item()
            running_unc += lu.item()
            all_pred_pres.append(torch.sigmoid(pres_logits).cpu())
            all_gt_pres.append(gt_pres.cpu())
            all_pred_expr.append(expr_mu.cpu())
            all_gt_expr.append(gt_expr.cpu())
            all_uncertainty.append(F.softplus(log_var).cpu())

    n_val = len(val_loader)
    val_total = running_loss / n_val
    val_loss_list.append(val_total)
    val_pres_loss_list.append(running_pres / n_val)
    val_expr_loss_list.append(running_expr / n_val)
    val_pcc_loss_list.append(running_pcc / n_val)
    val_unc_loss_list.append(running_unc / n_val)

    all_pred_pres_t = torch.cat(all_pred_pres)
    all_gt_pres_t = torch.cat(all_gt_pres)
    all_pred_expr_t = torch.cat(all_pred_expr)
    all_gt_expr_t = torch.cat(all_gt_expr)
    all_unc_t = torch.cat(all_uncertainty)

    val_pcc_pres = pearson_corr_vector(all_pred_pres_t, all_gt_pres_t).mean().item()
    val_pcc_expr = pearson_corr_vector(all_pred_expr_t, all_gt_expr_t).mean().item()
    val_pcc_pres_list.append(val_pcc_pres)
    val_pcc_expr_list.append(val_pcc_expr)
    val_mean_unc_list.append(all_unc_t.mean().item())

    print(f'[{epoch:4d}] '
          f'Train: {train_loss_list[-1]:.4f} (BCE:{train_pres_loss_list[-1]:.4f} SL1:{train_expr_loss_list[-1]:.4f} PCC:{train_pcc_loss_list[-1]:.4f} Unc:{train_unc_loss_list[-1]:.4f}) | '
          f'Val: {val_loss_list[-1]:.4f} (BCE:{val_pres_loss_list[-1]:.4f} SL1:{val_expr_loss_list[-1]:.4f} PCC:{val_pcc_loss_list[-1]:.4f} Unc:{val_unc_loss_list[-1]:.4f}) | '
          f'PCC_pres: {val_pcc_pres:.4f} PCC_expr: {val_pcc_expr:.4f} | '
          f'σ: {val_mean_unc_list[-1]:.4f}')

    # --- Checkpoint ---
    if val_total < MIN_loss and not np.isnan(val_total):
        MIN_loss = val_total
        torch.save(model.state_dict(), f'{model_path}train_3head_best.pt')

    # --- Plot ---
    if epoch % 100 == 99:
        fig, axes = plt.subplots(1, 4, figsize=(28, 5))

        ax = axes[0]
        ax.plot(train_loss_list, label='Train Total')
        ax.plot(val_loss_list, label='Val Total')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Total Loss'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[1]
        ax.plot(train_pres_loss_list, label='Train BCE(pres)', alpha=0.7)
        ax.plot(train_expr_loss_list, label='Train SL1(expr)', alpha=0.7, linestyle='--')
        ax.plot(val_pres_loss_list, label='Val BCE(pres)', alpha=0.7)
        ax.plot(val_expr_loss_list, label='Val SL1(expr)', alpha=0.7, linestyle='--')
        ax.plot(train_pcc_loss_list, label='Train PCC', alpha=0.7, linestyle='-.')
        ax.plot(val_pcc_loss_list, label='Val PCC', alpha=0.7, linestyle='-.')
        ax.plot(train_unc_loss_list, label='Train Calib(unc)', alpha=0.7, linestyle=':')
        ax.plot(val_unc_loss_list, label='Val Calib(unc)', alpha=0.7, linestyle=':')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Per-Head Loss'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

        ax = axes[2]
        ax.plot(train_pcc_pres_list, label='Train Pres PCC', alpha=0.7)
        ax.plot(val_pcc_pres_list, label='Val Pres PCC', alpha=0.7)
        ax.plot(train_pcc_expr_list, label='Train Expr PCC', alpha=0.7, linestyle='--')
        ax.plot(val_pcc_expr_list, label='Val Expr PCC', alpha=0.7, linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('PCC')
        ax.set_title('Pearson Correlation'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[3]
        ax.plot(train_mean_unc_list, label='Train Mean σ', alpha=0.7)
        ax.plot(val_mean_unc_list, label='Val Mean σ', alpha=0.7)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Mean Uncertainty (σ)')
        ax.set_title('Learned Uncertainty'); ax.legend(); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

print(f'\nBest val loss: {MIN_loss:.6f}')
print(f'Model saved: {model_path}train_3head_best.pt')

In [ ]:
# ===== Evaluation =====
model.load_state_dict(torch.load(f'{model_path}train_3head_best.pt', map_location=device))
model.eval()

all_pred_pres, all_gt_pres = [], []
all_pred_expr, all_gt_expr = [], []
all_uncertainty = []

with torch.no_grad():
    for img_20x, img_5x, gt_pres, gt_expr in tqdm(val_loader, desc='Evaluation', leave=True):
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        pres_logits, expr_mu, log_var = model(img_20x, img_5x)
        all_pred_pres.append(torch.sigmoid(pres_logits).cpu())
        all_gt_pres.append(gt_pres)
        all_pred_expr.append(expr_mu.cpu())
        all_gt_expr.append(gt_expr)
        all_uncertainty.append(F.softplus(log_var).cpu())  # calibrated σ

all_pred_pres = torch.cat(all_pred_pres)
all_gt_pres = torch.cat(all_gt_pres)
all_pred_expr = torch.cat(all_pred_expr)
all_gt_expr = torch.cat(all_gt_expr)
all_unc = torch.cat(all_uncertainty)

# Combined prediction: P(present) × E[expr | present]
all_pred_combined = all_pred_pres * all_pred_expr

# Per-gene metrics
pcc_pres = pearson_corr_vector(all_pred_pres, all_gt_pres)
pcc_expr = pearson_corr_vector(all_pred_expr, all_gt_expr)
pcc_combined = pearson_corr_vector(all_pred_combined, all_gt_expr)
mae_pres = (all_pred_pres - all_gt_pres).abs().mean(dim=0)
mae_expr = (all_pred_expr - all_gt_expr).abs().mean(dim=0)
mae_combined = (all_pred_combined - all_gt_expr).abs().mean(dim=0)

# Presence accuracy (threshold=0.5)
pres_binary = (all_pred_pres > 0.5).float()
pres_acc = (pres_binary == all_gt_pres).float().mean(dim=0)

# Mean uncertainty per gene
mean_unc = all_unc.mean(dim=0)

print(f'\n{"Gene":>10s} | {"Pres_Acc":>8s} {"PCC_Pres":>9s} | {"PCC_Expr":>9s} {"MAE_Expr":>9s} | {"PCC_Comb":>9s} {"MAE_Comb":>9s} | {"Mean_σ":>8s}')
print('-' * 100)
for gi, gene in enumerate(MARKER_GENES):
    print(f'{gene:>10s} | {pres_acc[gi]:8.4f} {pcc_pres[gi]:9.4f} | '
          f'{pcc_expr[gi]:9.4f} {mae_expr[gi]:9.4f} | '
          f'{pcc_combined[gi]:9.4f} {mae_combined[gi]:9.4f} | {mean_unc[gi]:8.4f}')
print('-' * 100)
print(f'{"Mean":>10s} | {pres_acc.mean():8.4f} {pcc_pres.mean():9.4f} | '
      f'{pcc_expr.mean():9.4f} {mae_expr.mean():9.4f} | '
      f'{pcc_combined.mean():9.4f} {mae_combined.mean():9.4f} | {mean_unc.mean():8.4f}')

# ===== Bar Chart: Per-Gene PCC =====
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
x = np.arange(NUM_GENES)
w = 0.25

ax = axes[0]
ax.bar(x, pcc_pres.numpy(), w, label='Presence PCC', color='steelblue')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('PCC'); ax.set_title('Presence: Pearson Correlation')
ax.legend(); ax.grid(True, alpha=0.3, axis='y'); ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(x - w/2, pcc_expr.numpy(), w, label='Expression PCC', color='coral')
ax.bar(x + w/2, pcc_combined.numpy(), w, label='Combined PCC', color='mediumpurple')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('PCC'); ax.set_title('Expression & Combined: PCC')
ax.legend(); ax.grid(True, alpha=0.3, axis='y'); ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[2]
ax.bar(x, mean_unc.numpy(), color='goldenrod')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('Mean σ'); ax.set_title('Learned Uncertainty per Gene')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# ===== Scatter Plot: Pred vs GT =====
fig, axes = plt.subplots(4, 5, figsize=(25, 20))
fig.suptitle('Expression: Predicted vs Ground Truth (combined = P × μ)', fontsize=16, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_expr[:, gi].numpy()
    pred_vals = all_pred_combined[:, gi].numpy()
    unc_vals = all_unc[:, gi].numpy()
    ax.scatter(gt_vals, pred_vals, alpha=0.15, s=8, c=unc_vals, cmap='coolwarm', vmin=0)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, alpha=0.7)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    ax.set_title(f'{gene}\nPCC={pcc_combined[gi]:.3f} σ={mean_unc[gi]:.3f}', fontsize=10)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
# Hide last unused subplot
axes[3, 4].axis('off')
plt.tight_layout()
plt.show()

# ===== Error Distribution Boxplot =====
error_combined = (all_pred_combined - all_gt_expr).numpy()

fig, ax = plt.subplots(figsize=(16, 6))
bp = ax.boxplot([error_combined[:, gi] for gi in range(NUM_GENES)],
                labels=MARKER_GENES, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('mediumpurple')
    patch.set_alpha(0.6)
ax.axhline(y=0, color='red', linewidth=0.8, linestyle='--')
ax.set_ylabel('Prediction Error (Pred - GT)')
ax.set_title('Combined Prediction Error Distribution (P × μ)')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# ===== Gene Group Summary =====
gene_groups = {
    'Epithelial/Tumor': [0, 1],
    'Stromal/Fibroblast': [2, 3, 4, 5],
    'Endothelial': [6],
    'Macrophage / Myeloid': [7, 8, 9, 10],
    'T Cell': [11, 12, 13, 14],
    'B Cell': [15, 16, 17, 18]
}

group_names = list(gene_groups.keys())
group_pcc_pres = [pcc_pres[idx].mean().item() for idx in gene_groups.values()]
group_pcc_comb = [pcc_combined[idx].mean().item() for idx in gene_groups.values()]
group_unc = [mean_unc[idx].mean().item() for idx in gene_groups.values()]

fig, axes = plt.subplots(1, 3, figsize=(21, 5))
gx = np.arange(len(group_names))
gw = 0.35

ax = axes[0]
ax.bar(gx - gw/2, group_pcc_pres, gw, label='Presence', color='steelblue')
ax.bar(gx + gw/2, group_pcc_comb, gw, label='Combined', color='mediumpurple')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean PCC'); ax.set_title('PCC by Gene Group')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(gx, group_unc, color='goldenrod')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean σ'); ax.set_title('Uncertainty by Gene Group')
ax.grid(True, alpha=0.3, axis='y')

# Presence accuracy by group
group_acc = [pres_acc[idx].mean().item() for idx in gene_groups.values()]
ax = axes[2]
ax.bar(gx, group_acc, color='seagreen')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Accuracy'); ax.set_title('Presence Accuracy by Gene Group')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== Uncertainty Analysis =====
# 불확실성이 높은 샘플 vs 낮은 샘플에서 실제 오차 비교

abs_error = (all_pred_combined - all_gt_expr).abs()  # (N, 19)
mean_unc_per_sample = all_unc.mean(dim=1)  # (N,)
mean_err_per_sample = abs_error.mean(dim=1)  # (N,)

# Uncertainty vs Error 상관관계
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(mean_unc_per_sample.numpy(), mean_err_per_sample.numpy(), alpha=0.1, s=8, color='steelblue')
ax.set_xlabel('Mean Uncertainty (σ)')
ax.set_ylabel('Mean Absolute Error')
ax.set_title('Uncertainty vs Prediction Error (sample-level)')
ax.grid(True, alpha=0.3)
# PCC
pcc_unc_err = pearson_corr_vector(
    mean_unc_per_sample.unsqueeze(1), mean_err_per_sample.unsqueeze(1)
).item()
ax.text(0.05, 0.95, f'PCC = {pcc_unc_err:.3f}', transform=ax.transAxes,
        fontsize=14, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Per-gene: 불확실성 상위/하위 25% MAE 비교
ax = axes[1]
high_unc_mae = []
low_unc_mae = []
for gi in range(NUM_GENES):
    unc_g = all_unc[:, gi]
    err_g = abs_error[:, gi]
    q25 = unc_g.quantile(0.25)
    q75 = unc_g.quantile(0.75)
    low_unc_mae.append(err_g[unc_g <= q25].mean().item())
    high_unc_mae.append(err_g[unc_g >= q75].mean().item())

x = np.arange(NUM_GENES)
w = 0.35
ax.bar(x - w/2, low_unc_mae, w, label='Low σ (bottom 25%)', color='seagreen')
ax.bar(x + w/2, high_unc_mae, w, label='High σ (top 25%)', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('MAE'); ax.set_title('MAE: Low vs High Uncertainty Samples')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nUncertainty calibration: if model is well-calibrated,')
print('high uncertainty samples should have higher error (tomato bars > green bars).')

In [ ]:
# ===== Patch Image + GT vs Pred 비교 =====
np.random.seed(123)
n_show = 8
show_idx = np.random.choice(len(all_gt_pres), n_show, replace=False)

fig, axes = plt.subplots(n_show, 5, figsize=(35, 5 * n_show),
                         gridspec_kw={'width_ratios': [1, 1, 1.2, 1.5, 1.5]})
fig.suptitle('3-Head: Patch Image + GT vs Pred Comparison', fontsize=18, y=1.005)

for row, si in enumerate(show_idx):
    # 20x patch
    img_20x = np.array(Image.open(val_dataset.img_20x_paths[si]).convert('RGB'))
    ax = axes[row, 0]
    ax.imshow(img_20x)
    ax.set_title(f'Sample {si} — 20x', fontsize=10)
    ax.axis('off')

    # 5x context
    img_5x = np.array(Image.open(val_dataset.img_5x_paths[si]).convert('RGB'))
    ax = axes[row, 1]
    ax.imshow(img_5x)
    ax.set_title(f'Sample {si} — 5x', fontsize=10)
    ax.axis('off')

    # Presence: GT vs Pred
    xg = np.arange(NUM_GENES)
    ax = axes[row, 2]
    ax.barh(xg - 0.2, all_gt_pres[si].numpy(), 0.4, label='GT', color='steelblue', alpha=0.7)
    ax.barh(xg + 0.2, all_pred_pres[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(xg); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_xlim(0, 1); ax.set_title('Presence P(gene)', fontsize=10)
    ax.legend(fontsize=6, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')

    # Expression: GT vs Pred (combined)
    ax = axes[row, 3]
    ax.barh(xg - 0.2, all_gt_expr[si].numpy(), 0.4, label='GT', color='orange', alpha=0.7)
    ax.barh(xg + 0.2, all_pred_combined[si].numpy(), 0.4, label='Pred (P×μ)', color='crimson', alpha=0.7)
    ax.set_yticks(xg); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_xlim(0, 1); ax.set_title('Expression (combined)', fontsize=10)
    ax.legend(fontsize=6, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')

    # Uncertainty per gene
    ax = axes[row, 4]
    colors = ['seagreen' if all_unc[si, gi] < mean_unc[gi] else 'tomato' for gi in range(NUM_GENES)]
    ax.barh(xg, all_unc[si].numpy(), color=colors, alpha=0.7)
    ax.axvline(x=all_unc[si].mean().item(), color='gray', linestyle='--', linewidth=1)
    ax.set_yticks(xg); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_title(f'Uncertainty (σ, mean={all_unc[si].mean():.3f})', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()